# **SyllabusRAG**

## **1. Install all dependencies**

In [ ]:
!pip install -r requirements.txt

## **2. Build and save the knowledge base**

In [ ]:
import json
import os
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("📄 Loading syllabus...")

# Set paths for data and output
DATA_FOLDER = "training_data"
TEXT_DB_PATH = os.path.join(DATA_FOLDER, "knowledge_base.txt")
FAISS_INDEX_PATH = os.path.join(DATA_FOLDER, "faiss_index")

# Create folder if needed
os.makedirs(DATA_FOLDER, exist_ok=True)

# Try loading the syllabus file
try:
    with open('data/syllabus.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print("Oops! syllabus.json not found.")
    data = {"courses": []}

# Prepare course content
documents = []
all_course_content = []

for course in data.get('courses', []):
    course_code = course.get('course_code', '')
    course_title = course.get('course_title', '')

    # Build course description
    content = f"Course Code: {course_code}\nCourse Title: {course_title}\n"
    prereqs = course.get('prerequisites')
    content += f"Prerequisites: {', '.join(prereqs) if prereqs else 'None'}\n"
    objectives = course.get('course_objectives')
    if objectives:
        content += "Objectives:\n" + "\n".join([f"- {obj}" for obj in objectives]) + "\n"
    modules = course.get('modules')
    if modules:
        content += "Modules:\n" + "\n".join([f"- {mod.get('module_title', '')}" for mod in modules])

    all_course_content.append(content)

    # Create LangChain document
    doc = Document(page_content=content, metadata={"course_code": course_code, "course_title": course_title})
    documents.append(doc)

# Save all course content to a file
print(f"💾 Saving course details to {TEXT_DB_PATH}...")
with open(TEXT_DB_PATH, 'w', encoding='utf-8') as f:
    f.write("\n\n---\n\n".join(all_course_content))
print("   ...Done.")

# Create FAISS index
print(f"🧠 Creating FAISS index at {FAISS_INDEX_PATH}...")
if documents:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(documents, embeddings)
    vector_store.save_local(FAISS_INDEX_PATH)
    print("   ...Done.")
    print("\n✅ Knowledge base ready!")
else:
    print("⚠️ No courses found.")

## **3. Load existing knowledge base**

In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

def load_knowledge_base():
    """Load the FAISS index and its documents."""
    FAISS_INDEX_PATH = os.path.join("training_data", "faiss_index")

    print(f"🧠 Loading knowledge base from {FAISS_INDEX_PATH}...")

    if not os.path.exists(FAISS_INDEX_PATH):
        print(f"Error: '{FAISS_INDEX_PATH}' not found.")
        return None, None

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # Load FAISS index
    vector_store = FAISS.load_local(FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)

    # Extract documents from the vector store
    documents = [doc for doc in vector_store.docstore._dict.values()]

    print("✅ Knowledge base loaded.")
    return vector_store, documents

# Load knowledge base into variables
vector_store, documents = load_knowledge_base()

## **4. Load the pre-trained model**

In [ ]:
import torch
import re
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from google.colab import userdata

# Holds the text-generation pipeline
qa_pipe = None

def load_model():
    """Load the Gemma-7B Instruct model for text generation."""
    global qa_pipe
    print("🤖 Loading Gemma-7B Instruct model...")
    model_id = "google/gemma-7b-it"
    hf_token = userdata.get('HF_TOKEN')

    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto",
        token=hf_token
    )

    qa_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256, return_full_text=False)
    print("✅ Model ready.")

# Load the model
load_model()

## **5. Ask the questions**

In [ ]:
import re

def ask_question(question):
    """Search for context in the knowledge base and ask the model to answer."""
    global documents, vector_store, qa_pipe
    print(f"\n❓ Question: {question}")

    retrieved_doc = None
    # Look for a course code like BECE309L in the question
    code_match = re.search(r'\b[A-Z]{4}\d{3}[A-Z,E,P,L]?\b', question)

    # If a course code is found, try to match it with a document
    if code_match:
        found_code = code_match.group(0)
        print(f"   (Course code detected: {found_code})")
        for doc in documents:
            if doc.metadata.get('course_code') == found_code:
                retrieved_doc = doc
                break

    # If no code was found, use similarity search
    if not retrieved_doc:
        print("   (No course code found, performing similarity search...)")
        retrieved_docs = vector_store.similarity_search(question)
        if retrieved_docs:
            retrieved_doc = retrieved_docs[0]

    # If still no document is found, inform the user
    if not retrieved_doc:
        print("✅ Answer:\nI couldn't find anything relevant.")
        return

    context = retrieved_doc.page_content

    # Prepare the prompt with the context and question
    messages = [
        {
            "role": "user",
            "content": f"Use this context to answer.\n\nContext:\n{context}\n\nQuestion: {question}"
        }
    ]

    prompt = qa_pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = qa_pipe(prompt)

    # Output the model's response
    print("✅ Answer:")
    print(outputs[0]['generated_text'].strip())

# Example questions
ask_question("What is the title for BECE309L")
ask_question("What are the prerequisites for BECE309L?")
ask_question("What are the objectives of the Artificial Intelligence and Machine Learning course?")